# A/B Testing Activation Calculation

This notebook reads raw `USERS_RAW` and `TRACKS_RAW` tables from Snowflake into pandas.

Then we do the A/B assignment and activation calculation in pandas.

Activation means the user completed both `workspace_created` and `report_generated` within 7 days of `first_logged_in_at`.

In [10]:
from pathlib import Path
import os

import pandas as pd
import snowflake.connector


def load_env(path: str = ".env") -> None:
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env()

,user_id,account_id,email,job_title,is_marketing_opted_in,created_at,first_logged_in_at,latest_logged_in_at,experience_in_years
0,e1e6cd80-18c9-41ab-bdef-1863593e373f,ffe675e9-5dbd-4959-bd4c-354eb8a0dc36,genevra.boeck@hotmail.fr,DevOps Engineer,0,2024-11-17 07:54:35,2024-11-26 16:37:35,2025-10-16 16:43:37,4
1,683ece6f-329f-4ece-977d-c0c0ca84068e,ffe675e9-5dbd-4959-bd4c-354eb8a0dc36,marlow.mulholland@yahoo.com,Network Engineer,0,2024-11-05 23:12:48,2024-11-12 02:20:19,2025-09-25 07:41:53,1
2,15e51e79-d39d-4019-8f3f-1b9fcbe32720,ffe675e9-5dbd-4959-bd4c-354eb8a0dc36,amie.horstead@hotmail.com,Cloud Engineer,0,2025-05-12 00:02:32,2025-05-19 18:48:48,2025-06-14 21:47:51,9
3,73075a63-19cd-4aa4-bf42-d9e3ccb3d2de,ffe675e9-5dbd-4959-bd4c-354eb8a0dc36,sancho.derricoat@gmail.com,Technical Support Specialist,0,2024-11-06 09:22:31,2024-11-12 07:20:36,2024-12-04 22:02:21,8
4,92df33f0-eb2e-4570-ac5f-febcbde3c047,ffe675e9-5dbd-4959-bd4c-354eb8a0dc36,nerita.jakes@hotmail.com,Graphic Designer,1,2024-06-08 09:56:37,2024-06-09 12:49:37,2025-01-18 15:54:16,8
...,...,...,...,...,...,...,...,...,...
6001,7f2a5c8d-4b9e-4673-8d1a-3f6c9e2b5d8a,f3e7c892-4d5a-4b2c-9a1e-7f8b3c4d5e6f,mike.rodriguez@atlasmanufacturing.com,Plant Supervisor,0,2024-10-22 17:45:51,2024-10-25 14:03:38,2025-11-18 22:53:34,9
6002,9e4c7b1d-6a2f-4958-b3e7-2c5f8a4d7b9e,f3e7c892-4d5a-4b2c-9a1e-7f8b3c4d5e6f,jennifer.wong@atlasmanufacturing.com,Quality Assurance Lead,1,2024-06-09 20:29:06,2024-06-13 16:40:19,2025-11-21 19:18:45,20
6003,3c6f9a2d-8e1b-4237-9f4c-7a3d6e9b2c5f,2b8e4f7a-9c3d-4e5f-8a2b-1c7d9e4f6a8b,david.thompson@verdeenergy.com,Senior Project Manager,1,2024-07-14 14:56:42,2024-07-16 19:33:30,2025-11-22 17:45:21,19
6004,8b5d2a7f-3c9e-4614-8a5b-4d7f2a9c6e3b,2b8e4f7a-9c3d-4e5f-8a2b-1c7d9e4f6a8b,lisa.patel@verdeenergy.com,Systems Analyst,0,2024-09-03 22:39:57,2024-09-05 15:27:05,2025-11-19 21:02:49,20


In [9]:
import hashlib

connection_params = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USERNAME"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"],
}

if os.getenv("SNOWFLAKE_ROLE"):
    connection_params["role"] = os.environ["SNOWFLAKE_ROLE"]

if os.getenv("SNOWFLAKE_AUTHENTICATOR"):
    connection_params["authenticator"] = os.environ["SNOWFLAKE_AUTHENTICATOR"]

with snowflake.connector.connect(**connection_params) as conn:
    users = pd.read_sql("SELECT * FROM USERS_RAW", conn)
    tracks = pd.read_sql("SELECT * FROM TRACKS_RAW", conn)

users.columns = users.columns.str.lower()
tracks.columns = tracks.columns.str.lower()

users["first_logged_in_at"] = pd.to_datetime(users["first_logged_in_at"])
tracks["event_timestamp"] = pd.to_datetime(tracks["event_timestamp"])

def assign_variant(user_id: str) -> str:
    hash_value = int(hashlib.md5(user_id.encode()).hexdigest(), 16)
    return "treatment" if hash_value % 2 else "control"

assignments = users[["user_id"]].copy()
assignments["variant"] = assignments["user_id"].apply(assign_variant)

activation_events = tracks[tracks["event_name"].isin(["workspace_created", "report_generated"])]
activation_events = activation_events.merge(
    users[["user_id", "first_logged_in_at"]],
    on="user_id",
    how="inner",
)

activation_events = activation_events[
    (activation_events["event_timestamp"] >= activation_events["first_logged_in_at"])
    & (activation_events["event_timestamp"] <= activation_events["first_logged_in_at"] + pd.Timedelta(days=7))
]

user_activation = (
    activation_events
    .pivot_table(
        index="user_id",
        columns="event_name",
        values="event_timestamp",
        aggfunc="min",
    )
    .reset_index()
)

user_activation["is_activated"] = (
    user_activation["workspace_created"].notna()
    & user_activation["report_generated"].notna()
)

experiment_users = users[["user_id", "first_logged_in_at"]].merge(assignments, on="user_id")
experiment_users = experiment_users.merge(
    user_activation[["user_id", "is_activated"]],
    on="user_id",
    how="left",
)
experiment_users["is_activated"] = experiment_users["is_activated"].fillna(False).astype(bool)

summary = (
    experiment_users
    .groupby("variant", as_index=False)
    .agg(
        users=("user_id", "count"),
        activated_users=("is_activated", "sum"),
    )
)
summary["activation_rate"] = summary["activated_users"] / summary["users"]
summary

/var/folders/35/mcj68sf96mz81d075p4nxy3m0000gp/T/ipykernel_88055/2416322428.py:54: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  experiment_users["is_activated"] = experiment_users["is_activated"].fillna(False)


,variant,users,activated_users,activation_rate
0,control,2977,78,0.026201
1,treatment,3029,80,0.026411
